In [ ]:
import pandas as pd
import numpy as np
from .dates import year_fraction
from .import_data import load_market, MarketEngine
from .vanilla_option import OptionType, VanillaOptionBlackSholes, d, strike_for_delta
from .binary_option import CoNBlackScholes, AoNBlackScholes

class PricingEngine:
    def __init__(self, market_file: str, tenor: str, imply_pln: bool):
        self.market = load_market(market_file, tenor)
        self.m_engine = MarketEngine(self.market)

        self.r_pln, self.r_eur, self.df_d, self.df_f = (self.m_engine.rates_dfs(imply_pln))

        self.sigma_atm = self.market.atm
        self.sigma_25C = self.market.atm + self.market.bf25 + 0.5 * self.market.rr25
        self.sigma_25P = self.market.atm + self.market.bf25 - 0.5 * self.market.rr25
        
    def strike(self,delta: float,option_type: OptionType):
        sigma = self.sigma_atm
        return strike_for_delta(
            delta=delta,
            option_type=option_type,
            df_d=self.df_d,
            df_f=self.df_f,
            S_t=self.market.spot,
            sigma=sigma,
            t=self.market.start,
            T=self.market.expiry,
            base=365,
            forward=self.market.delta_forward,
            premium=self.market.delta_premium,
        )
         
    def vanilla_price(self,strike: float,option_type: OptionType,pricing_method: str,p: float = 1.0,):
        option = VanillaOptionBlackSholes(T=self.market.expiry,K=strike,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def con_price(self,strike: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001,):
        option = CoNBlackScholes(T=self.market.expiry, K=strike, option_type=option_type )
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )
        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def aon_price(self,strike: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001):
        option = AoNBlackScholes(T=self.market.expiry,K=strike,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("pricing_method must be 'bs' or 'vv'")